<a href="https://colab.research.google.com/github/donleaveher/plasticity-placement/blob/agent%2Fadd-lora-evaluation/notebooks/p0c_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P0-C reproducible workflow: Smoke → Calibration → Pilot

This notebook is the canonical entry point for the early P0-C workflow. It pins experiment code and model revisions, separates incompatible runtime environments automatically, streams progress, persists logs to Drive, verifies every manifest, and blocks invalid downstream phases.

**Do not use `Run all` to launch GPU experiments.** Run setup once, then enable exactly one phase switch at a time.

## Execution rules

1. Select a GPU runtime before setup.
2. Leave `REQUESTED_CODE_REVISION = None` to use the current branch head, or set an exact Git SHA to reproduce an earlier run.
3. A normal disconnect resumes the same phase attempt when code, model, configuration, GPU, CUDA, Python, and locked packages match.
4. If a manifest contains an immutable `failed` unit, preserve it and increment only that phase's attempt label (`a1` → `a2`).
5. Never edit `calibration_report.json`, `manifest.json`, or compiled artifacts by hand.
6. Confirmatory is intentionally kept in `p0c_confirmatory_colab.ipynb`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
import os
import re
import shlex
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path

from IPython.display import display

# Repository selection. Set an exact SHA for a historical reproduction.
REPO_URL = 'https://github.com/donleaveher/plasticity-placement.git'
BRANCH = 'agent/add-lora-evaluation'
REQUESTED_CODE_REVISION = None
REPO_DIR = Path('/content/plasticity-placement')

# Frozen experiment settings shared by Smoke, Calibration, and Pilot.
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
REQUESTED_MODEL_REVISION = None
USE_4BIT = True
MAX_LENGTH = 256
MAX_NEW_TOKENS = 8

# Increment only the failed phase. Keep the old directory for audit.
SMOKE_ATTEMPT = 'a1'
CALIBRATION_ATTEMPT = 'a1'
PILOT_ATTEMPT = 'a1'

DRIVE_BASE = Path('/content/drive/MyDrive/plasticity-p0c/runs')

for name, value in {
    'SMOKE_ATTEMPT': SMOKE_ATTEMPT,
    'CALIBRATION_ATTEMPT': CALIBRATION_ATTEMPT,
    'PILOT_ATTEMPT': PILOT_ATTEMPT,
}.items():
    if not re.fullmatch(r'[A-Za-z0-9._-]+', value):
        raise ValueError(f'{name} contains unsafe path characters: {value!r}')

## 1. Checkout and install the frozen environment

This cell refuses to overwrite a dirty repository. Colab's `/content` checkout is disposable; experiment artifacts live only under Drive.

In [ ]:
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)

if REPO_DIR.exists() and not (REPO_DIR / '.git').exists():
    raise RuntimeError(f'{REPO_DIR} exists but is not a Git repository')
if not REPO_DIR.exists():
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )

dirty = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'status', '--porcelain'],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if dirty:
    raise RuntimeError(
        'The Colab repository has local changes; start a fresh runtime or save them '
        f'before continuing:\n{dirty}'
    )

subprocess.run(
    ['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH],
    check=True,
)
revision_ref = REQUESTED_CODE_REVISION or f'origin/{BRANCH}'
CODE_REVISION = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', f'{revision_ref}^{{commit}}'],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
subprocess.run(
    ['git', '-C', str(REPO_DIR), 'checkout', '--detach', CODE_REVISION],
    check=True,
)
subprocess.run(
    ['uv', 'sync', '--extra', 'train', '--extra', 'colab'],
    cwd=REPO_DIR,
    check=True,
)
print('Checked out:', CODE_REVISION)

## 2. Resolve provenance and derive safe Drive directories

The output root is derived from formal experiment code, the resolved model revision, experiment settings, and the critical runtime environment. Notebook prose/layout changes do not invalidate formal code provenance; source or dependency changes do.

In [ ]:
def run_json(command):
    completed = subprocess.run(
        command,
        cwd=REPO_DIR,
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        print(completed.stdout)
        print(completed.stderr)
        raise RuntimeError(
            f'Command failed with exit code {completed.returncode}: '
            f'{shlex.join(command)}'
        )
    lines = [line for line in completed.stdout.splitlines() if line.strip()]
    if not lines:
        raise RuntimeError(f'Command returned no JSON: {shlex.join(command)}')
    return json.loads(lines[-1])

resolve_command = [
    'uv', 'run', 'plasticity-p0c', 'resolve-model',
    '--model', MODEL_NAME,
]
if REQUESTED_MODEL_REVISION is not None:
    resolve_command.extend(['--model-revision', REQUESTED_MODEL_REVISION])
model_context = run_json(resolve_command)
MODEL_REVISION = model_context['resolved_revision']
if not MODEL_REVISION:
    raise RuntimeError('Could not resolve an immutable model revision')

environment_context = run_json([
    'uv', 'run', 'plasticity-p0c', 'environment',
])
environment = environment_context['environment']
ENVIRONMENT_FINGERPRINT = environment_context['fingerprint']
CODE_HASH = environment['code_sha256']

experiment_settings = {
    'model_name': MODEL_NAME,
    'model_revision': MODEL_REVISION,
    'use_4bit': USE_4BIT,
    'max_length': MAX_LENGTH,
    'max_new_tokens': MAX_NEW_TOKENS,
}
settings_payload = json.dumps(
    experiment_settings,
    sort_keys=True,
    separators=(',', ':'),
)
SETTINGS_FINGERPRINT = sha256(settings_payload.encode()).hexdigest()[:10]
PROVENANCE_KEY = (
    f'code-{CODE_HASH[:10]}_cfg-{SETTINGS_FINGERPRINT}_'
    f'env-{ENVIRONMENT_FINGERPRINT}'
)
PROVENANCE_ROOT = DRIVE_BASE / PROVENANCE_KEY
SMOKE_DIR = PROVENANCE_ROOT / f'smoke-{SMOKE_ATTEMPT}'
CALIBRATION_DIR = PROVENANCE_ROOT / f'calibration-{CALIBRATION_ATTEMPT}'
PILOT_DIR = PROVENANCE_ROOT / f'pilot-{PILOT_ATTEMPT}'
LOG_DIR = PROVENANCE_ROOT / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

critical_environment = {
    key: environment.get(key)
    for key in (
        'python', 'packages', 'cuda_available', 'cuda_version', 'gpu', 'code_sha256'
    )
}
frozen_context = {
    'schema_version': 1,
    'code_sha256': CODE_HASH,
    'environment_fingerprint': ENVIRONMENT_FINGERPRINT,
    'critical_environment': critical_environment,
    'experiment_settings': experiment_settings,
}
context_path = PROVENANCE_ROOT / 'run_context.json'
if context_path.exists():
    existing_context = json.loads(context_path.read_text())
    if existing_context != frozen_context:
        raise RuntimeError(f'Frozen run context mismatch: {context_path}')
else:
    context_tmp = context_path.with_suffix('.json.tmp')
    context_tmp.write_text(
        json.dumps(frozen_context, indent=2, sort_keys=True) + '\n'
    )
    context_tmp.replace(context_path)

sessions_path = PROVENANCE_ROOT / 'source_sessions.json'
sessions = json.loads(sessions_path.read_text()) if sessions_path.exists() else []
sessions.append({
    'recorded_at': datetime.now(timezone.utc).isoformat(),
    'git_revision': CODE_REVISION,
    'branch': BRANCH,
})
sessions_tmp = sessions_path.with_suffix('.json.tmp')
sessions_tmp.write_text(json.dumps(sessions, indent=2, sort_keys=True) + '\n')
sessions_tmp.replace(sessions_path)

print('Git revision:', CODE_REVISION)
print('Formal code hash:', CODE_HASH)
print('Model revision:', MODEL_REVISION)
print('GPU:', environment.get('gpu'))
print('Environment fingerprint:', ENVIRONMENT_FINGERPRINT)
print('Provenance root:', PROVENANCE_ROOT)
print('Smoke:', SMOKE_DIR)
print('Calibration:', CALIBRATION_DIR)
print('Pilot:', PILOT_DIR)

## 3. Logging, recovery, and verification helpers

Every child process streams merged stdout/stderr into this notebook and a timestamped Drive log. On failure, the raised message points to the full log instead of hiding the root cause behind `CalledProcessError`.

In [ ]:
def run_checked(label, command):
    timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    safe_label = re.sub(r'[^A-Za-z0-9._-]+', '-', label)
    log_path = LOG_DIR / f'{timestamp}-{safe_label}.log'
    child_environment = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    started = time.monotonic()
    print(f'\n[{label}] $ {shlex.join(command)}')
    print(f'[{label}] log: {log_path}')
    with log_path.open('w', encoding='utf-8') as log_file:
        log_file.write(f'$ {shlex.join(command)}\n')
        log_file.flush()
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            env=child_environment,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        try:
            assert process.stdout is not None
            for line in process.stdout:
                print(line, end='')
                log_file.write(line)
                log_file.flush()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
        return_code = process.wait()
    elapsed = time.monotonic() - started
    print(f'[{label}] exit={return_code} elapsed={elapsed / 60:.1f} min')
    if return_code != 0:
        raise RuntimeError(
            f'{label} failed with exit code {return_code}. Full log: {log_path}'
        )
    return log_path

def read_json(path):
    return json.loads(Path(path).read_text())

def manifest_status(output_dir):
    manifest_path = Path(output_dir) / 'manifest.json'
    if not manifest_path.exists():
        return {
            'exists': False,
            'message': 'No manifest; failure occurred before or during initialization.',
        }
    manifest = read_json(manifest_path)
    return {
        'exists': True,
        'run_id': manifest.get('run_id'),
        'selected_lessons': manifest.get('selected_lessons', []),
        'base_states': dict(Counter(manifest.get('base_arms', {}).values())),
        'unit_states': dict(Counter(
            unit.get('state') for unit in manifest.get('units', {}).values()
        )),
        'errors': manifest.get('errors', [])[-10:],
    }

def verify_manifest(output_dir, expected_lessons, expected_units):
    manifest_path = Path(output_dir) / 'manifest.json'
    if not manifest_path.exists():
        raise FileNotFoundError(f'Missing manifest: {manifest_path}')
    manifest = read_json(manifest_path)
    selected = manifest.get('selected_lessons', [])
    base_arms = manifest.get('base_arms', {})
    units = manifest.get('units', {})
    if len(selected) != expected_lessons:
        raise RuntimeError(
            f'Expected {expected_lessons} lessons, observed {len(selected)}'
        )
    if len(base_arms) != expected_lessons or set(base_arms.values()) != {'verified'}:
        raise RuntimeError(f'Base arms are incomplete: {Counter(base_arms.values())}')
    if len(units) != expected_units:
        raise RuntimeError(f'Expected {expected_units} units, observed {len(units)}')
    bad_units = {
        name: unit.get('state')
        for name, unit in units.items()
        if unit.get('state') != 'verified'
        or float(unit.get('rollback_exact_match_rate', 0.0)) != 1.0
    }
    if bad_units:
        raise RuntimeError(f'Unverified units or rollback failures: {bad_units}')
    if manifest.get('errors'):
        raise RuntimeError(f'Manifest contains recorded errors: {manifest["errors"][-5:]}')
    print(
        f'Verified manifest: lessons={len(selected)}, units={len(units)}, rollback=1.0'
    )
    return manifest

def model_cli_args():
    arguments = [
        '--model', MODEL_NAME,
        '--model-revision', MODEL_REVISION,
        '--max-length', str(MAX_LENGTH),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
    ]
    if USE_4BIT:
        arguments.append('--use-4bit')
    return arguments

## 4. Tier 0 Smoke

Set `RUN_SMOKE = True` once. Re-running the same healthy attempt is a zero-work verification/resume. If an immutable failure is recorded, increment `SMOKE_ATTEMPT` in the configuration cell and rerun setup.

In [ ]:
RUN_SMOKE = False

if RUN_SMOKE:
    run_checked('smoke-run', [
        'uv', 'run', 'plasticity-p0c', 'run',
        '--output', str(SMOKE_DIR),
        '--tier', 'smoke',
        *model_cli_args(),
        '--rank', '8',
        '--alpha', '16',
        '--learning-rate', '2e-4',
        '--max-steps', '16',
    ])
    verify_manifest(SMOKE_DIR, expected_lessons=2, expected_units=2)
    run_checked('smoke-aggregate', [
        'uv', 'run', 'plasticity-p0c', 'aggregate',
        '--output', str(SMOKE_DIR),
        '--bootstrap-samples', '1000',
    ])

print(json.dumps(manifest_status(SMOKE_DIR), ensure_ascii=False, indent=2))
smoke_summary_path = SMOKE_DIR / 'results' / 'aggregate' / 'summary.json'
if smoke_summary_path.exists():
    smoke_summary = read_json(smoke_summary_path)
    display(smoke_summary['arm_summary'])
    display(smoke_summary['contrasts'])

## 5. Development Calibration

Calibration is blocked until Smoke has two verified units with exact rollback. It trains 60 development adapters. The CLI now prints each candidate and adapter transition directly. A completed report is terminal and will never be silently overwritten, including when `selected_config` is null.

In [ ]:
RUN_CALIBRATION = False
CALIBRATION_REPORT = CALIBRATION_DIR / 'calibration_report.json'

if RUN_CALIBRATION:
    verify_manifest(SMOKE_DIR, expected_lessons=2, expected_units=2)
    run_checked('calibration', [
        'uv', 'run', 'plasticity-p0c', 'calibrate',
        '--output', str(CALIBRATION_DIR),
        '--model', MODEL_NAME,
        '--model-revision', MODEL_REVISION,
        '--max-length', str(MAX_LENGTH),
        '--max-new-tokens', str(MAX_NEW_TOKENS),
        '--seed', '42',
        '--bootstrap-samples', '1000',
        *(['--use-4bit'] if USE_4BIT else []),
    ])

if CALIBRATION_REPORT.exists():
    calibration_report = read_json(CALIBRATION_REPORT)
    stage1_states = Counter(
        row.get('status') for row in calibration_report.get('stage1_results', [])
    )
    final_states = Counter(
        row.get('status') for row in calibration_report.get('final_results', [])
    )
    print('Stage 1:', stage1_states)
    print('Final:', final_states)
    print('Survivors:', calibration_report.get('survivor_ids'))
    print('Selected config:')
    print(json.dumps(calibration_report.get('selected_config'), indent=2))
else:
    print('No calibration report yet. Set RUN_CALIBRATION = True after Smoke passes.')

### Calibration diagnostics (read-only)

This cell distinguishes a scientific no-qualified-config result from orchestration failures. It never starts training.

In [ ]:
if not CALIBRATION_REPORT.exists():
    print('No calibration report to diagnose.')
else:
    calibration_report = read_json(CALIBRATION_REPORT)
    rows = []
    for result in calibration_report.get('final_results', []):
        metrics = result.get('combined') or {}
        failed_gates = []
        if metrics:
            if metrics['target_gain'] < 0.40:
                failed_gates.append('target_gain<0.40')
            if metrics['p_exact'] < 0.75:
                failed_gates.append('p_exact<0.75')
            if metrics['p_interference'] > 0.05:
                failed_gates.append('interference>0.05')
            if metrics['invalid_increase'] > 0.05:
                failed_gates.append('invalid_increase>0.05')
        rows.append({
            'candidate_id': result.get('candidate_id'),
            'status': result.get('status'),
            'qualified': result.get('qualified', False),
            'target_gain': metrics.get('target_gain'),
            'p_exact': metrics.get('p_exact'),
            'p_interference': metrics.get('p_interference'),
            'invalid_increase': metrics.get('invalid_increase'),
            'failed_gates': ', '.join(failed_gates) or ('PASS' if metrics else 'no metrics'),
            'stage2_error': result.get('stage2_error'),
        })
    if rows:
        import pandas as pd
        display(pd.DataFrame(rows))
    else:
        print('No final results. Stage-1 diagnostics:')
        for result in calibration_report.get('stage1_results', []):
            if result.get('status') != 'completed':
                print(result.get('candidate_id'), result.get('error'))

## 6. Eight-lesson Pilot

Pilot is blocked unless the frozen calibration report exists and contains a non-null `selected_config`. Screening may use predefined reserve pairs to satisfy the strict complete-pair and four-action balance constraints.

In [ ]:
RUN_PILOT = False

if RUN_PILOT:
    verify_manifest(SMOKE_DIR, expected_lessons=2, expected_units=2)
    if not CALIBRATION_REPORT.exists():
        raise FileNotFoundError(f'Missing calibration report: {CALIBRATION_REPORT}')
    calibration_report = read_json(CALIBRATION_REPORT)
    selected_config = calibration_report.get('selected_config')
    if not isinstance(selected_config, dict) or not selected_config:
        raise RuntimeError(
            'Calibration has no qualified selected_config. Increment the calibration '
            'attempt only after changing the preregistered training backend.'
        )
    run_checked('pilot-run', [
        'uv', 'run', 'plasticity-p0c', 'run',
        '--output', str(PILOT_DIR),
        '--tier', 'pilot',
        *model_cli_args(),
        '--calibration-config', str(CALIBRATION_REPORT),
    ])
    verify_manifest(PILOT_DIR, expected_lessons=8, expected_units=8)
    run_checked('pilot-aggregate', [
        'uv', 'run', 'plasticity-p0c', 'aggregate',
        '--output', str(PILOT_DIR),
        '--bootstrap-samples', '10000',
    ])

print(json.dumps(manifest_status(PILOT_DIR), ensure_ascii=False, indent=2))
pilot_summary_path = PILOT_DIR / 'results' / 'aggregate' / 'summary.json'
if pilot_summary_path.exists():
    pilot_summary = read_json(pilot_summary_path)
    display(pilot_summary['arm_summary'])
    display(pilot_summary['contrasts'])
    display(pilot_summary['pair_action_diagnostics'])
    display(pilot_summary['action_token_diagnostics'])

    lesson_contrasts = pilot_summary['lesson_contrasts']
    positive_pn = sum(
        row['target_difference'] > 0
        for row in lesson_contrasts
        if row['contrast'] == 'P-N'
    )
    nontrivial_pe = sum(
        abs(row['target_difference']) >= 0.10
        or abs(row['interference_difference']) >= 0.10
        for row in lesson_contrasts
        if row['contrast'] == 'P-E'
    )
    p_summary = pilot_summary['arm_summary']['parametric']
    print('Pilot preregistered gate evidence:')
    print('  P>N lessons:', positive_pn, '/ 8 (requires >= 6)')
    print('  Nontrivial P/E lessons:', nontrivial_pe, '/ 8 (requires >= 2)')
    print('  P mean interference:', p_summary['interference_regression'])
    print('  P invalid rate:', p_summary['invalid_rate'])
    print('Review pair/action diagnostics before declaring Tier-1 GO.')

## 7. Recovery and next stage

- To inspect a failure, read the Drive log path printed by `run_checked`, then inspect `manifest_status(...)`.
- Resume the same attempt only after a normal disconnect and only when no unit is `failed`.
- If a unit is `failed`, preserve the directory and increment that phase's attempt label.
- Code, dependency, model, precision, length, CUDA, or GPU changes automatically produce a different provenance root.
- A completed calibration report is terminal. A null selection is a valid no-go result, not permission to edit the report.
- After Pilot passes, use the standalone [`p0c_confirmatory_colab.ipynb`](p0c_confirmatory_colab.ipynb).